In [1]:
from pyspark.sql import SparkSession
import sys
import os

In [2]:
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")    
    .getOrCreate()
)

In [3]:
df = spark.read.csv("data/cleaned/df_train.csv", header=True, inferSchema=True)
test = spark.read.csv("data/cleaned/df_test.csv", header=True, inferSchema=True)

In [4]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_dow: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- store_and_fwd_flag_index: double (nullable = true)
 |-- log_trip_duration: double (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [5]:
test.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_dow: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- store_and_fwd_flag_index: double (nullable = true)



In [6]:
df.show(5)

+---------+---------+---------------+------------+-----------+----------+----------+------------------+------------------------+------------------+-------------+
|       id|vendor_id|passenger_count|pickup_month|pickup_hour|pickup_dow|is_weekend|       distance_km|store_and_fwd_flag_index| log_trip_duration|trip_duration|
+---------+---------+---------------+------------+-----------+----------+----------+------------------+------------------------+------------------+-------------+
|id2875421|        2|              1|           3|         17|         2|         0|1.4985207796469109|                     0.0|6.1224928095143865|          455|
|id2377394|        1|              1|           6|          0|         1|         1|1.8055071687958897|                     0.0| 6.498282149476434|          663|
|id3858529|        2|              1|           1|         11|         3|         0| 6.385098495252496|                     0.0| 7.661527081358517|         2124|
|id3504673|        2|       

In [7]:
df.columns

['id',
 'vendor_id',
 'passenger_count',
 'pickup_month',
 'pickup_hour',
 'pickup_dow',
 'is_weekend',
 'distance_km',
 'store_and_fwd_flag_index',
 'log_trip_duration',
 'trip_duration']

In [8]:
from pyspark.ml.feature import VectorAssembler

vector = VectorAssembler(
    inputCols=['vendor_id', 'passenger_count', 'pickup_month', 'pickup_hour', 'pickup_dow', 'is_weekend', 'distance_km', 'store_and_fwd_flag_index'],
    outputCol='features'
)

final_df = vector.transform(df)
test = vector.transform(test)

In [9]:
final_df.show(5)

+---------+---------+---------------+------------+-----------+----------+----------+------------------+------------------------+------------------+-------------+--------------------+
|       id|vendor_id|passenger_count|pickup_month|pickup_hour|pickup_dow|is_weekend|       distance_km|store_and_fwd_flag_index| log_trip_duration|trip_duration|            features|
+---------+---------+---------------+------------+-----------+----------+----------+------------------+------------------------+------------------+-------------+--------------------+
|id2875421|        2|              1|           3|         17|         2|         0|1.4985207796469109|                     0.0|6.1224928095143865|          455|[2.0,1.0,3.0,17.0...|
|id2377394|        1|              1|           6|          0|         1|         1|1.8055071687958897|                     0.0| 6.498282149476434|          663|[1.0,1.0,6.0,0.0,...|
|id3858529|        2|              1|           1|         11|         3|         0| 

In [10]:
test.show(5)

+---------+---------+---------------+------------+-----------+----------+----------+------------------+------------------------+--------------------+
|       id|vendor_id|passenger_count|pickup_month|pickup_hour|pickup_dow|is_weekend|       distance_km|store_and_fwd_flag_index|            features|
+---------+---------+---------------+------------+-----------+----------+----------+------------------+------------------------+--------------------+
|id3004672|        1|              1|           6|         23|         5|         0| 2.746425820747344|                     0.0|[1.0,1.0,6.0,23.0...|
|id3505355|        1|              1|           6|         23|         5|         0|2.7592389329009714|                     0.0|[1.0,1.0,6.0,23.0...|
|id1217141|        1|              1|           6|         23|         5|         0|1.3061553897970135|                     0.0|[1.0,1.0,6.0,23.0...|
|id2150126|        2|              1|           6|         23|         5|         0|5.26908774072880

In [11]:
final_df = final_df.select('features', 'log_trip_duration', 'trip_duration')
test = test.select('features')

In [12]:
train, val = final_df.randomSplit([0.75, 0.25], seed=42)


In [13]:
print("Train: ", train.count(), len(train.columns))
print("Validation: ", val.count(), len(val.columns))
print("Test: ", test.count(), len(test.columns))

Train:  1092603 3
Validation:  363929 3
Test:  625134 1


# Linear Regression

In [14]:
from pyspark.ml.regression import LinearRegression

model = LinearRegression(featuresCol='features', labelCol='log_trip_duration')

model = model.fit(train)

In [15]:
model.coefficients

DenseVector([0.0067, 0.0076, 0.018, 0.0052, 0.0153, -0.1025, 0.1046, 0.0138])

In [16]:
model.intercept

5.909290235041533

In [17]:
pred_results = model.evaluate(val)

In [18]:
pred_results.predictions.show()

+--------------------+------------------+-------------+-----------------+
|            features| log_trip_duration|trip_duration|       prediction|
+--------------------+------------------+-------------+-----------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23|6.069511383622059|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362|5.994260012632986|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284|6.094292015385203|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871|6.081738485248286|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624|6.168160697734506|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50|5.924629048823074|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321|5.941926759630917|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180|5.975137435147538|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183|5.980888713574563|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317|5.997017740840732|
|[1.0,1.0,1.0,0.0,...| 6.2576675878826

In [19]:
pred_results.r2, pred_results.meanAbsoluteError, pred_results.meanSquaredError

(0.3736911981011358, 0.44702683930380954, 0.3769422467367252)

In [20]:
pred_results.rootMeanSquaredError

0.6139562254238695

In [21]:
from pyspark.sql import functions as F

rmsle = pred_results.predictions.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("log_trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.09008971434652249


In [22]:
from pyspark.ml.regression import LinearRegression

model = LinearRegression(featuresCol='features', labelCol='trip_duration')

model = model.fit(train)

In [23]:
model.coefficients

DenseVector([2.4546, 4.6455, 19.0335, 3.4408, 12.1477, -100.0161, 106.631, 86.3855])

In [24]:
model.intercept

322.63455164042153

In [25]:
pred_results = model.evaluate(val)

In [26]:
pred_results.predictions.show()

+--------------------+------------------+-------------+------------------+
|            features| log_trip_duration|trip_duration|        prediction|
+--------------------+------------------+-------------+------------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23| 459.7215462763683|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362| 387.6659528816117|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284|481.20969101238154|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871| 475.9478263315069|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624| 542.5044935172563|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50| 332.4143966425869|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321| 350.0413244095343|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180|383.88407880207393|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183| 389.7448178632693|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317|406.18081923536954|
|[1.0,1.0,1.0,0.0,...| 6.

In [27]:
pred_results.r2, pred_results.meanAbsoluteError, pred_results.meanSquaredError

(0.5336897427681782, 296.7113330731022, 202557.98693008884)

In [28]:
from pyspark.sql import functions as F

rmsle = pred_results.predictions.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.5934774590407262


# Ridge

In [29]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

ridge = LinearRegression(
    featuresCol="features",
    labelCol="log_trip_duration",
    predictionCol="prediction",
    regParam=0.1,       
    elasticNetParam=0.0,  
    maxIter=100,
    standardization=True
)


model = ridge.fit(train)

pred_results = model.evaluate(val)
rmse = pred_results.rootMeanSquaredError
print("RMSE =", rmse)


RMSE = 0.6188186234950398


In [30]:
from pyspark.sql import functions as F

rmsle = pred_results.predictions.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("log_trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.09137663951705302


In [31]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

ridge = LinearRegression(
    featuresCol="features",
    labelCol="trip_duration",
    predictionCol="prediction",
    regParam=1.0,       
    elasticNetParam=0.0,  
    maxIter=100,
    standardization=True
)


model = ridge.fit(train)

pred_results = model.evaluate(val)
rmse = pred_results.rootMeanSquaredError
print("RMSE =", rmse)


RMSE = 450.11586228304816


In [32]:
pred_results.predictions.show()

+--------------------+------------------+-------------+------------------+
|            features| log_trip_duration|trip_duration|        prediction|
+--------------------+------------------+-------------+------------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23| 460.2663416742007|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362|388.34791048016746|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284| 481.7435129816271|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871| 476.5132852008907|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624| 542.9763632666638|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50| 333.1939832686705|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321|350.79408949583575|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180| 384.5853479728489|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183|390.43716919833514|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317| 406.8481611717194|
|[1.0,1.0,1.0,0.0,...| 6.

In [33]:
from pyspark.sql import functions as F

rmsle = pred_results.predictions.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.59375528690181


In [34]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

evaluator = RegressionEvaluator(labelCol="trip_duration", predictionCol="prediction", metricName="rmse")

paramGrid = (ParamGridBuilder()
    .addGrid(ridge.regParam, [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0])
    .build()
)

cv = CrossValidator(
    estimator=Pipeline(stages=[ridge]),
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4
)

cvModel = cv.fit(train)
bestModel = cvModel.bestModel

bestRegParam = bestModel.stages[-1]._java_obj.getRegParam()
print("Best regParam =", bestRegParam)

pred = bestModel.transform(val)
rmse = evaluator.evaluate(pred)
print("Test RMSE =", rmse)


Best regParam = 10.0
Test RMSE = 450.6168649579488


# Decision Tree

In [35]:
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="trip_duration",
    predictionCol="prediction",
    maxDepth=8,      
    minInstancesPerNode=20
)

dt_model = dt.fit(train)
pred = dt_model.transform(val)

evaluator = RegressionEvaluator(
    labelCol="trip_duration",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(pred)
print("RMSE =", rmse)


RMSE = 361.0511723233161


In [36]:
from pyspark.sql import functions as F

rmsle = pred.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.4680821805811411


In [37]:
pred.show()

+--------------------+------------------+-------------+------------------+
|            features| log_trip_duration|trip_duration|        prediction|
+--------------------+------------------+-------------+------------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23|236.27313600599476|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362|236.27313600599476|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284|236.27313600599476|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871|236.27313600599476|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624|236.27313600599476|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50| 269.2874776676953|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321| 269.2874776676953|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180| 423.5777678171224|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183| 423.5777678171224|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317| 423.5777678171224|
|[1.0,1.0,1.0,0.0,...| 6.

# Random Forest

In [38]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_duration",
    predictionCol="prediction",
    numTrees=50,         
    maxDepth=10,         
    minInstancesPerNode=50,
    subsamplingRate=0.7, 
    featureSubsetStrategy="auto",
    seed=42
)

rf_model = rf.fit(train)
pred = rf_model.transform(val)  


In [39]:
for metric in ["rmse", "mae", "r2"]:
    evaluator = RegressionEvaluator(
        labelCol="trip_duration",
        predictionCol="prediction",
        metricName=metric
    )
    print(metric.upper(), "=", evaluator.evaluate(pred))


RMSE = 359.7863481412428
MAE = 236.51486084314024
R2 = 0.7020008969309048


In [40]:
from pyspark.sql import functions as F

rmsle = pred.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.48175374820472344


In [41]:
pred.show()

+--------------------+------------------+-------------+------------------+
|            features| log_trip_duration|trip_duration|        prediction|
+--------------------+------------------+-------------+------------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23|259.22628388197364|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362| 294.4022378801256|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284| 294.6678257073702|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871|303.63568036237257|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624| 308.2375857215636|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50|309.60010339767115|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321| 321.3532838229257|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180| 382.7986245878346|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183| 382.7986245878346|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317| 445.0330972096152|
|[1.0,1.0,1.0,0.0,...| 6.

# Boosting

In [42]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="trip_duration",
    predictionCol="prediction",
    maxIter=50,      
    maxDepth=10,         
    stepSize=0.05,        
    subsamplingRate=0.7,  
    seed=42
)

gbt_model = gbt.fit(train)
pred = gbt_model.transform(val)  




In [43]:
for metric in ["rmse", "mae", "r2"]:
    val_metric = RegressionEvaluator(
        labelCol="trip_duration",
        predictionCol="prediction",
        metricName=metric
    ).evaluate(pred)
    print(metric.upper(), "=", val_metric)


RMSE = 354.4500910314756
MAE = 230.93071209060207
R2 = 0.7107750326167048


In [44]:
from pyspark.sql import functions as F

rmsle = pred.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.45991568159164475


In [45]:
pred.show()

+--------------------+------------------+-------------+------------------+
|            features| log_trip_duration|trip_duration|        prediction|
+--------------------+------------------+-------------+------------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23|218.58329436929785|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362| 281.1351615663913|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284|251.24117004569308|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871| 381.3124235834528|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624| 518.3222750361605|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50|281.97758165049214|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321|297.74854916227264|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180| 393.5020259534113|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183| 393.5020259534113|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317|447.80225688171896|
|[1.0,1.0,1.0,0.0,...| 6.

# Generalized Linear Regression

In [46]:
from pyspark.ml.regression import GeneralizedLinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

glr = GeneralizedLinearRegression(
    featuresCol="features",
    labelCol="trip_duration",   
    predictionCol="prediction",
    family="gamma",           
    link="log",             
    regParam=0.0,
    maxIter=50
)

glr_model = glr.fit(train)
pred = glr_model.transform(val)



In [47]:
for metric in ["rmse", "mae", "r2"]:
    val_metric = RegressionEvaluator(
        labelCol="trip_duration",
        predictionCol="prediction",
        metricName=metric
    ).evaluate(pred)
    print(metric.upper(), "=", val_metric)


RMSE = 1.7022857073993907e+31
MAE = 3.319340052479699e+28
R2 = -6.670993259935434e+56


In [48]:
from pyspark.sql import functions as F

rmsle = pred.select(
    F.sqrt(
        F.mean(
            F.pow(
                F.log1p("trip_duration") - F.log1p("prediction"), 2
            )
        )
    ).alias("RMSLE")
).collect()[0]["RMSLE"]

print(rmsle)


0.6330552176914374


In [49]:
pred.show()

+--------------------+------------------+-------------+------------------+
|            features| log_trip_duration|trip_duration|        prediction|
+--------------------+------------------+-------------+------------------+
|(8,[0,1,2,4],[1.0...|3.1780538303479458|           23| 477.4203318113749|
|(8,[0,1,2,4],[2.0...| 7.217443431696533|         1362|435.90116730824144|
|(8,[0,1,2,4],[2.0...| 5.652489180268651|          284| 487.1935405784105|
|(8,[0,1,2,4],[2.0...|  6.77078942390898|          871| 480.5748806023545|
|(8,[0,1,2,4],[2.0...| 7.393263094763838|         1624| 521.9824430503659|
|[1.0,1.0,1.0,0.0,...|3.9318256327243257|           50|406.85257498892713|
|[1.0,1.0,1.0,0.0,...|5.7745515455444085|          321| 415.3711270453367|
|[1.0,1.0,1.0,0.0,...| 5.198497031265826|          180| 432.2294501347793|
|[1.0,1.0,1.0,0.0,...| 5.214935757608986|          183| 435.2176412301413|
|[1.0,1.0,1.0,0.0,...| 5.762051382780177|          317| 443.7084630222563|
|[1.0,1.0,1.0,0.0,...| 6.